# Fine-tune LFM2.5-VL-1.6B on UCF Crime for Sentinel tier-3

Adapted from Yashraj Maher's LFM-UCF project (https://yashrajmaher.com/my-words/lfm-ucf)
and Unsloth's LFM2.5-VL guide (https://unsloth.ai/docs/models/tutorials/lfm2.5).

**This notebook has not been run.** It was written from the blog post's
description and Unsloth's documented API, not tested against a live
session (this project's sandbox has huggingface.co blocked, so there was
no way to verify it end to end). If a cell errors on an API mismatch,
cross-check against Unsloth's actual published LFM2.5-VL notebook, linked
above, that one is the tested reference.

**Drive is mounted first, before anything else.** This session's own
weapons-detector training lost a full run to a Colab disconnect that wiped
`/content` before that lesson was learned. Not repeating it here.

Target: fine-tune so the model can answer `backend/threat.py`'s actual
`threat_prompt()` format (a one-sentence description + an `INCIDENT:`
line), not a standalone JSON schema — see `training/vlm/README.md` for why.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = '/content/drive/MyDrive/sentinel_lfm_ucf'
import os
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(f'{RUN_DIR}/checkpoints', exist_ok=True)

In [ ]:
!pip install -q unsloth datasets pillow

import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'no GPU attached — Runtime > Change runtime type > T4 GPU, then Runtime > Restart session'

## Dataset

Run `training/vlm/prepare_ucf_dataset.py` first (in this same Colab, or
anywhere with internet) to produce `train.jsonl` / `holdout.jsonl` plus an
`images/` folder. Point `DATASET_DIR` at wherever that landed — ideally
already inside Drive so it survives a disconnect too.

In [ ]:
DATASET_DIR = f'{RUN_DIR}/ucf_prepared'  # output of prepare_ucf_dataset.py

# If prepare_ucf_dataset.py hasn't been run yet, do it here:
# !pip install -q datasets huggingface_hub
# !python -m training.vlm.prepare_ucf_dataset --out {DATASET_DIR}

import json
from pathlib import Path
from PIL import Image

def load_jsonl(path):
    return [json.loads(l) for l in open(path)]

train_records = load_jsonl(f'{DATASET_DIR}/train.jsonl')
holdout_records = load_jsonl(f'{DATASET_DIR}/holdout.jsonl')
print(f'{len(train_records)} train, {len(holdout_records)} holdout')
print(train_records[0])

In [ ]:
# Convert to the chat-format Unsloth's vision trainer expects: one
# {"messages": [...]} conversation per example, image + prompt as the user
# turn, the templated INCIDENT-line target as the assistant turn.

def to_conversation(rec, dataset_dir):
    img = Image.open(Path(dataset_dir) / rec['image']).convert('RGB')
    return {
        'messages': [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': rec['prompt']},
            ]},
            {'role': 'assistant', 'content': [
                {'type': 'text', 'text': rec['response']},
            ]},
        ]
    }

train_conversations = [to_conversation(r, DATASET_DIR) for r in train_records]
print(f'built {len(train_conversations)} training conversations')

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name='LiquidAI/LFM2.5-VL-1.6B',
    max_seq_length=2048,
    load_in_4bit=False,
)

# Freeze vision layers (already knows what things look like — no need to
# retrain the vision tower), fine-tune language layers only (teach it
# surveillance-frame vocabulary and the INCIDENT: output format). Same
# recipe the blog used: rank 16, ~0.5-1% of parameters trained.
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
)

In [ ]:
# Checkpoint + resume: free Colab gives ~4.5h T4 time per 24h, this run
# needs more. Checkpoints land in Drive, so a disconnect costs minutes, not
# the whole run — the exact problem the weapons-detector training hit
# earlier this session, fixed here from the start instead of after losing
# a run to it.

from trl import SFTTrainer, SFTConfig
import glob, os

CKPT_DIR = f'{RUN_DIR}/checkpoints'
existing_ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'),
                        key=lambda p: int(p.split('-')[-1]))
resume_from = existing_ckpts[-1] if existing_ckpts else None
print('resuming from:', resume_from or '(fresh start)')

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_conversations,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        optim='adamw_8bit',
        save_strategy='steps',
        save_steps=100,
        save_total_limit=3,
        output_dir=CKPT_DIR,
        report_to='none',
    ),
)

trainer.train(resume_from_checkpoint=resume_from)

## Export

Two forms: the merged HF checkpoint (for further fine-tuning or transformers-based inference) and a GGUF export (for local llama.cpp/Ollama inference on your Mac — matches what `backend/lfm_vlm.py` expects and what the blog itself published).

In [ ]:
OUT_DIR = f'{RUN_DIR}/lfm_ucf_merged'
model.save_pretrained_merged(OUT_DIR, tokenizer, save_method='merged_16bit')
print('merged checkpoint saved to', OUT_DIR)

# GGUF export for local CPU inference (llama.cpp / Ollama):
try:
    model.save_pretrained_gguf(f'{RUN_DIR}/lfm_ucf.gguf', tokenizer, quantization_method='q4_k_m')
    print('GGUF saved to', f'{RUN_DIR}/lfm_ucf.gguf')
except Exception as exc:
    print('GGUF export failed (verify against Unsloth docs — API may differ for vision models):', exc)

## Next

1. Download the GGUF (or merged checkpoint) from Drive.
2. Place it at `checkpoints/lfm_ucf.gguf` in the repo.
3. Run `python -m training.vlm.evaluate --model checkpoints/lfm_ucf.gguf` locally — against this project's own labelled footage, not UCF Crime. That is the number that decides whether this is usable.
4. Only if that passes: `export SENTINEL_VLM_BACKEND=lfm` and `export SENTINEL_LFM_MODEL=checkpoints/lfm_ucf.gguf`.